In [11]:
import random
from qiskit import QuantumCircuit
from qiskit_aer import Aer
# NOTE: The 'execute' function is deprecated. We will use the simulator's .run() method instead.
from qiskit.visualization import plot_histogram

# Use the high-performance Aer simulator
simulator = Aer.get_backend('aer_simulator')

# --- Simulation Parameters ---
NUM_BITS = 100 # The total number of qubits Alice will send

# 1. Alice generates her random classical data
alice_bits = [random.randint(0, 1) for _ in range(NUM_BITS)]
# 0 for Rectilinear (+), 1 for Diagonal (x)
alice_bases = [random.randint(0, 1) for _ in range(NUM_BITS)] 

print("--- Alice's Setup ---")
print(f"Bits to send:  {alice_bits[:10]}...")
print(f"Bases for bits:{alice_bases[:10]}... (0:+, 1:x)")

def alice_encodes(bits, bases):
    """Creates a list of QuantumCircuits based on Alice's bits and bases."""
    qubits = []
    for i in range(len(bits)):
        # Create a circuit with 1 qubit and 1 classical bit
        qc = QuantumCircuit(1, 1)
        
        # Encode the bit '1' by applying an X-gate
        if bits[i] == 1:
            qc.x(0)
            
        # Encode in the diagonal basis by applying an H-gate
        if bases[i] == 1:
            qc.h(0)
        
        qubits.append(qc)
    return qubits

alice_qubits = alice_encodes(alice_bits, alice_bases)

# Let's inspect the first qubit circuit as an example
print("\nExample: Circuit for Alice's first qubit:")
print(f"Bit: {alice_bits[0]}, Basis: {'x' if alice_bases[0] else '+'}")
print(alice_qubits[0].draw(output='text'))

# Bob generates his random bases
bob_bases = [random.randint(0, 1) for _ in range(NUM_BITS)]
print("\n--- Bob's Setup ---")
print(f"Bases for measurement: {bob_bases[:10]}... (0:+, 1:x)")

def bob_measures(qubits, bases):
    """Measures a list of qubits in the bases specified."""
    measured_bits = []
    # Create a copy of the circuits to avoid modifying the original list
    qubits_to_measure = [qc.copy() for qc in qubits]

    for i in range(len(qubits_to_measure)):
        qc = qubits_to_measure[i]
        
        # Apply H-gate if Bob wants to measure in the diagonal basis
        if bases[i] == 1:
            qc.h(0)
            
        # Measure the qubit
        qc.measure(0, 0)
        
        # *** FIX IS HERE ***
        # Execute the circuit on the simulator using the modern backend.run() method
        # We run each circuit for 1 shot since it's a single measurement
        job = simulator.run(qc, shots=1, memory=True) # Using memory=True gives a list of results
        result = job.result()
        
        # The result is a list of strings, e.g., ['0'] or ['1'].
        measured_bit = int(result.get_memory(qc)[0])
        measured_bits.append(measured_bit)
        
    return measured_bits

bob_results = bob_measures(alice_qubits, bob_bases)
print(f"Bob's measured results: {bob_results[:10]}...")
  
def sift_keys(alice_bases, bob_bases, alice_bits, bob_bits):
    """Alice and Bob compare bases and keep only the matching bits."""
    sifted_key_alice = []
    sifted_key_bob = []
    for i in range(NUM_BITS):
        if alice_bases[i] == bob_bases[i]:
            sifted_key_alice.append(alice_bits[i])
            sifted_key_bob.append(bob_bits[i])
            
    return sifted_key_alice, sifted_key_bob

sifted_alice, sifted_bob = sift_keys(alice_bases, bob_bases, alice_bits, bob_results)

print("\n--- Post-Processing: Sifting ---")
print(f"Alice's sifted key: {sifted_alice[:20]}...")
print(f"Bob's sifted key:   {sifted_bob[:20]}...")
print(f"Length of sifted key: {len(sifted_alice)}")

def check_for_eavesdropper(key_a, key_b, sample_size=20):
    """Compare a sample of the keys to calculate the error rate (QBER)."""
    errors = 0
    # Ensure sample size is not larger than the key itself
    sample_size = min(sample_size, len(key_a))
    if sample_size == 0:
        return 0.0, [] # No bits to compare
    
    # Choose random indices to compare
    sample_indices = random.sample(range(len(key_a)), sample_size)
    sample_indices.sort(reverse=True) # Sort to remove items from the end first
    
    for i in sample_indices:
        if key_a[i] != key_b[i]:
            errors += 1
            
    qber = errors / sample_size if sample_size > 0 else 0.0
    
    # Create the final key by removing the publicly revealed sample bits
    final_key_a = list(key_a)
    for i in sample_indices:
        final_key_a.pop(i)
        
    return qber, final_key_a

qber, final_key = check_for_eavesdropper(sifted_alice, sifted_bob)

print("\n--- Security Check ---")
print(f"Quantum Bit Error Rate (QBER): {qber:.2%}")
if qber > 0.05: # Setting a 5% error threshold
    print("High QBER detected! Eavesdropping likely. ABORT.")
else:
    print("QBER is acceptable. A shared secret key is established.")
    print(f"Final shared key length: {len(final_key)}")
    print(f"Final key: {final_key[:20]}...")

def eve_intercepts_and_resends(qubits):
    """Eve measures each qubit and prepares a new one to send to Bob."""
    eve_bases = [random.randint(0, 1) for _ in range(len(qubits))]
    
    # Eve measures Alice's qubits
    eve_measured_bits = bob_measures(qubits, eve_bases)
    
    # Eve prepares and sends new qubits to Bob based on her measurements
    eves_forwarded_qubits = alice_encodes(eve_measured_bits, eve_bases)
    
    return eves_forwarded_qubits
  
def run_bb84_simulation(eavesdropper_present=False):
    print(f"\n{'='*50}")
    print(f"RUNNING SIMULATION - Eavesdropper Present: {eavesdropper_present}")
    print(f"{'='*50}\n")
    
    # 1. Alice prepares her data and qubits
    alice_bits = [random.randint(0, 1) for _ in range(NUM_BITS)]
    alice_bases = [random.randint(0, 1) for _ in range(NUM_BITS)]
    alice_qubits = alice_encodes(alice_bits, alice_bases)
    
    # Decide which qubits Bob will receive
    if eavesdropper_present:
        # 2a. Eve intercepts, measures, and resends
        print("--- Eve is intercepting the channel! ---")
        qubits_for_bob = eve_intercepts_and_resends(alice_qubits)
    else:
        # 2b. Channel is secure
        print("--- Channel is secure ---")
        qubits_for_bob = alice_qubits
        
    # 3. Bob generates bases and measures
    bob_bases = [random.randint(0, 1) for _ in range(NUM_BITS)]
    bob_results = bob_measures(qubits_for_bob, bob_bases)
    
    # 4. Sifting
    sifted_alice, sifted_bob = sift_keys(alice_bases, bob_bases, alice_bits, bob_results)
    
    # 5. Security Check
    qber, final_key = check_for_eavesdropper(sifted_alice, sifted_bob)
    
    print("\n--- SIMULATION RESULTS ---")
    print(f"Initial bits sent by Alice: {NUM_BITS}")
    print(f"Bits remaining after sifting: {len(sifted_alice)}")
    print(f"Quantum Bit Error Rate (QBER): {qber:.2%}")
    
    if qber > 0.05:
        print("CONCLUSION: High error rate detected! Eavesdropper is caught. Key is discarded.")
    else:
        print("CONCLUSION: No significant eavesdropping detected. Secure key established.")
        print(f"Final shared key length: {len(final_key)}")

# --- Run Both Scenarios ---
run_bb84_simulation(eavesdropper_present=False)
run_bb84_simulation(eavesdropper_present=True)

--- Alice's Setup ---
Bits to send:  [0, 1, 0, 0, 0, 0, 0, 1, 1, 1]...
Bases for bits:[1, 0, 0, 0, 0, 1, 1, 1, 1, 1]... (0:+, 1:x)

Example: Circuit for Alice's first qubit:
Bit: 0, Basis: x
     ┌───┐
  q: ┤ H ├
     └───┘
c: 1/═════
          

--- Bob's Setup ---
Bases for measurement: [0, 0, 0, 1, 0, 0, 1, 0, 0, 0]... (0:+, 1:x)
Bob's measured results: [0, 1, 0, 1, 0, 0, 0, 1, 1, 0]...

--- Post-Processing: Sifting ---
Alice's sifted key: [1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0]...
Bob's sifted key:   [1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0]...
Length of sifted key: 43

--- Security Check ---
Quantum Bit Error Rate (QBER): 0.00%
QBER is acceptable. A shared secret key is established.
Final shared key length: 23
Final key: [1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1]...

RUNNING SIMULATION - Eavesdropper Present: False

--- Channel is secure ---

--- SIMULATION RESULTS ---
Initial bits sent by Alice: 100
Bits remaining after si